<a href="https://colab.research.google.com/github/vidhanj2107-bot/Diabetes-Prediction-Model-mini-project-/blob/main/Diabetes_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install lightgbm boruta imbalanced-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              precision_recall_curve, average_precision_score,
                              ConfusionMatrixDisplay, f1_score, roc_curve, auc)
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import RandomOverSampler
from boruta import BorutaPy
import lightgbm as lgb


In [ ]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
           "Insulin", "BMI", "DiabetesPedigree", "Age", "Outcome"]

df = pd.read_csv(url, names=columns)

print("Original shape:", df.shape)
df.head()

Original shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigree,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## Figure 1: Dataset Overview Visualizations
Statistical summary heatmap, zero value counts, and target variable distribution.

In [ ]:
# Figure 1a/1b/1c: Dataset Overview Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Statistical summary heatmap
desc = df.describe().T
sns.heatmap(desc[['mean', 'std', 'min', '25%', '50%', '75%', 'max']],
            annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0])
axes[0].set_title('Figure 1a: Statistical Summary Heatmap')

# Zero/missing value bar chart
zero_cols_check = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
zero_counts = (df[zero_cols_check] == 0).sum()
axes[1].bar(zero_counts.index, zero_counts.values, color='steelblue')
axes[1].set_title('Figure 1b: Zero Value Counts per Column')
axes[1].set_xlabel('Column')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
for bar, val in zip(axes[1].patches, zero_counts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 str(val), ha='center', fontsize=9)

# Target variable distribution
target_counts = df['Outcome'].value_counts().sort_index()
axes[2].bar(['No Diabetes (0)', 'Diabetes (1)'], target_counts.values,
            color=['steelblue', 'salmon'])
axes[2].set_title('Figure 1c: Target Variable Distribution')
axes[2].set_ylabel('Count')
for bar, val in zip(axes[2].patches, target_counts.values):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
                 str(val), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('fig1_overview.png', dpi=300)
plt.show()


In [ ]:
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

df[zero_cols] = df[zero_cols].replace(0, np.nan)

## Figure 2: Missing Value Visualizations
Heatmap of missing values and bar chart of missing percentages.

In [ ]:
# Figure 2a/2b: Missing Value Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Missing value heatmap
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False, ax=axes[0])
axes[0].set_title('Figure 2a: Missing Value Heatmap (Yellow = Missing)')

# Missing percentage bar chart
miss_pct = df.isnull().mean() * 100
miss_pct = miss_pct[miss_pct > 0]
axes[1].bar(miss_pct.index, miss_pct.values, color='tomato')
axes[1].set_title('Figure 2b: Missing Value Percentage per Column')
axes[1].set_xlabel('Column')
axes[1].set_ylabel('Missing %')
axes[1].tick_params(axis='x', rotation=45)
for bar, val in zip(axes[1].patches, miss_pct.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig2_missing.png', dpi=300)
plt.show()


In [ ]:
imputer = SimpleImputer(strategy="mean")
df[zero_cols] = imputer.fit_transform(df[zero_cols])

## Figure 3: Before vs After Imputation
Box plot comparison and pairplot of key features.

In [ ]:
# Figure 3a/3b: Before vs After Imputation Comparison
cols_imputed = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df_before = df.copy()
# Re-read original to get pre-imputation data for comparison
zero_cols_nb = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
df_before[cols_imputed].boxplot(ax=axes[0])
axes[0].set_title('Figure 3a: Before Imputation (with NaN)')
axes[0].tick_params(axis='x', rotation=45)

df_after_imp = df.copy()
from sklearn.impute import SimpleImputer as _SI
_imp = _SI(strategy='mean')
df_after_imp[cols_imputed] = _imp.fit_transform(df_after_imp[cols_imputed])
df_after_imp[cols_imputed].boxplot(ax=axes[1])
axes[1].set_title('Figure 3b: After Mean Imputation')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Before vs After Mean Imputation', fontsize=13)
plt.tight_layout()
plt.savefig('fig3_imputation_comparison.png', dpi=300)
plt.show()

# Figure 3c: Pairplot of key features
key_cols = ['Glucose', 'BMI', 'Age', 'DiabetesPedigree', 'Outcome']
pairplot_df = df_after_imp[key_cols].copy()
pairplot_df['Outcome'] = pairplot_df['Outcome'].astype(int)
pair_fig = sns.pairplot(pairplot_df, hue='Outcome',
                        palette={0: 'steelblue', 1: 'salmon'},
                        plot_kws={'alpha': 0.5})
pair_fig.fig.suptitle('Figure 3c: Pairplot of Key Features by Outcome', y=1.02)
pair_fig.savefig('fig3c_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
multiplier = 1.3 # stricter than 1.5

Q1 = X_temp.quantile(0.25)
Q3 = X_temp.quantile(0.75)
IQR = Q3 - Q1

mask = ~((X_temp < (Q1 - multiplier * IQR)) |
         (X_temp > (Q3 + multiplier * IQR))).any(axis=1)

X_clean = X_temp[mask]
y_clean = y_temp[mask]

print("After Strict IQR:", X_clean.shape)

After Strict IQR: (446, 8)


## Figure 4: Outlier Removal Analysis
Box plots before/after IQR cleaning, row count reduction, and outcome distribution.

In [ ]:
# Figure 4a: Box Plots Before vs After Outlier Removal
cols_vis = ['Glucose', 'BMI', 'Age', 'BloodPressure']
df_clean_nb = df[df.notna().all(axis=1)].copy()  # use imputed df

# Use df (after imputation) as 'before' and X_clean as 'after'
fig, axes = plt.subplots(2, len(cols_vis), figsize=(16, 8))
for i, col in enumerate(cols_vis):
    df[col].plot(kind='box', ax=axes[0, i])
    axes[0, i].set_title(f'{col}\n(Before)')
    X_clean[col].plot(kind='box', ax=axes[1, i])
    axes[1, i].set_title(f'{col}\n(After)')
axes[0, 0].set_ylabel('Before IQR Removal')
axes[1, 0].set_ylabel('After IQR Removal')
plt.suptitle('Figure 4a: Box Plots Before vs After IQR Outlier Removal', fontsize=13)
plt.tight_layout()
plt.savefig('fig4_outlier_boxplots.png', dpi=300)
plt.show()

# Figure 4b/4c: Data Size Reduction and Outcome Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
bar_vals = [len(df), len(X_clean)]
bar_labels = ['Before Cleaning', 'After Cleaning']
bars = axes[0].bar(bar_labels, bar_vals, color=['steelblue', 'green'])
axes[0].set_title('Figure 4b: Row Count Before vs After IQR Cleaning')
axes[0].set_ylabel('Number of Rows')
for bar, val in zip(bars, bar_vals):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
                 str(val), ha='center', fontsize=11)

outcome_counts = y_clean.value_counts().sort_index()
axes[1].pie(outcome_counts.values, labels=['No Diabetes', 'Diabetes'],
            autopct='%1.1f%%', colors=['steelblue', 'salmon'], startangle=90)
axes[1].set_title('Figure 4c: Outcome Distribution After Cleaning')
plt.tight_layout()
plt.savefig('fig4bc_cleaning_summary.png', dpi=300)
plt.show()


In [72]:
X = df_clean.drop("Outcome", axis=1)
y = df_clean["Outcome"]

In [ ]:
ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X, y)

print("Class distribution after oversampling:")
print(pd.Series(y_res).value_counts())

Class distribution after oversampling:
Outcome
1    336
0    336
Name: count, dtype: int64


## Figure 5: Class Balance Visualizations
Class distribution before and after oversampling.

In [ ]:
# Figure 5a/5b/5c: Class Balance Before vs After Oversampling
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

before_counts = y.value_counts().sort_index()
after_counts_os = pd.Series(y_res).value_counts().sort_index()

axes[0].bar(['No Diabetes (0)', 'Diabetes (1)'], before_counts.values,
            color=['steelblue', 'salmon'])
axes[0].set_title('Figure 5a: Before Oversampling')
axes[0].set_ylabel('Count')
for bar, val in zip(axes[0].patches, before_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 str(val), ha='center', fontsize=10)

axes[1].bar(['No Diabetes (0)', 'Diabetes (1)'], after_counts_os.values,
            color=['steelblue', 'salmon'])
axes[1].set_title('Figure 5b: After Oversampling')
axes[1].set_ylabel('Count')
for bar, val in zip(axes[1].patches, after_counts_os.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 str(val), ha='center', fontsize=10)

axes[2].pie(after_counts_os.values, labels=['No Diabetes', 'Diabetes'],
            autopct='%1.1f%%', colors=['steelblue', 'salmon'], startangle=90)
axes[2].set_title('Figure 5c: Balanced Class Distribution')

plt.suptitle('Figure 5: Class Distribution Before vs After Oversampling', fontsize=13)
plt.tight_layout()
plt.savefig('fig5_oversample.png', dpi=300)
plt.show()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res,
    test_size=0.3,
    stratify=y_res,
    random_state=42
)

## Figure 6: Train/Test Split Visualization
Stacked bar and pie charts showing split and class distribution.

In [ ]:
# Figure 6a/6b/6c: Train/Test Split Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Stacked bar chart
axes[0].bar(['Dataset'], [len(X_train)], label='Train', color='steelblue')
axes[0].bar(['Dataset'], [len(X_test)], bottom=[len(X_train)], label='Test', color='salmon')
axes[0].set_title('Figure 6a: Train/Test Split Size')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].text(0, len(X_train) / 2, f'Train\n{len(X_train)}',
             ha='center', color='white', fontsize=11)
axes[0].text(0, len(X_train) + len(X_test) / 2, f'Test\n{len(X_test)}',
             ha='center', color='white', fontsize=11)

# Train class distribution pie
train_counts = pd.Series(y_train).value_counts().sort_index()
axes[1].pie(train_counts.values, labels=['Class 0', 'Class 1'],
            autopct='%1.1f%%', colors=['steelblue', 'salmon'], startangle=90)
axes[1].set_title('Figure 6b: Train Set Class Distribution')

# Test class distribution pie
test_counts = pd.Series(y_test).value_counts().sort_index()
axes[2].pie(test_counts.values, labels=['Class 0', 'Class 1'],
            autopct='%1.1f%%', colors=['steelblue', 'salmon'], startangle=90)
axes[2].set_title('Figure 6c: Test Set Class Distribution')

plt.tight_layout()
plt.savefig('fig6_split.png', dpi=300)
plt.show()


In [ ]:
rf = RandomForestClassifier(
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)

boruta = BorutaPy(
    rf,
    n_estimators='auto',
    max_iter=100,
    random_state=42
)

boruta.fit(X_train.values, y_train.values)

selected_features = X_train.columns[boruta.support_].tolist()
print("Selected Features:", selected_features)

Selected Features: ['Glucose', 'BMI', 'DiabetesPedigree', 'Age']


In [ ]:
rf = RandomForestClassifier(
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)

boruta = BorutaPy(
    rf,
    n_estimators='auto',
    max_iter=100,
    random_state=42
)

boruta.fit(X_train.values, y_train.values)

selected_features = X_train.columns[boruta.support_].tolist()
print("Selected Features:", selected_features)

Selected Features: ['Glucose', 'BMI', 'DiabetesPedigree', 'Age']


In [ ]:
lgb_model = lgb.LGBMClassifier(
    random_state=42,
    n_estimators=800,
    learning_rate=0.025,
    num_leaves=35,
    min_child_samples=10,
    subsample=0.9,
    colsample_bytree=0.9
)

lgb_model.fit(X_train[selected_features], y_train)

y_pred = lgb_model.predict(X_test[selected_features])

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

[LightGBM] [Info] Number of positive: 235, number of negative: 235
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000065 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 382
[LightGBM] [Info] Number of data points in the train set: 470, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

## Figure 7: Final Model Comprehensive Analysis
Precision-Recall curves, classification report heatmap, and summary dashboard.

In [ ]:
# Figure 7a: Precision-Recall Curves
plt.figure(figsize=(8, 6))
y_prob = lgb_model.predict_proba(X_test[selected_features])[:, 1]
prec, rec, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
plt.plot(rec, prec, label=f'LightGBM (AP={ap:.4f})', color='steelblue')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Figure 7a: Precision-Recall Curve')
plt.legend()
plt.tight_layout()
plt.savefig('fig7a_pr_curve.png', dpi=300)
plt.show()

# Figure 7b: ROC Curve
plt.figure(figsize=(8, 6))
fpr_nb, tpr_nb, _ = roc_curve(y_test, y_prob)
plt.plot(fpr_nb, tpr_nb, label=f'LightGBM (AUC={auc(fpr_nb, tpr_nb):.4f})', color='steelblue')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Figure 7b: ROC Curve')
plt.legend()
plt.tight_layout()
plt.savefig('fig7b_roc.png', dpi=300)
plt.show()

# Figure 7c: Classification Report Heatmap
report = classification_report(y_test, y_pred,
                               target_names=['No Diabetes', 'Diabetes'],
                               output_dict=True)
report_df = pd.DataFrame(report).T.drop(columns=['support'], errors='ignore')
report_df = report_df.loc[['No Diabetes', 'Diabetes', 'macro avg', 'weighted avg']]
plt.figure(figsize=(8, 4))
sns.heatmap(report_df.astype(float), annot=True, fmt='.3f',
            cmap='Blues', vmin=0, vmax=1)
plt.title('Figure 7c: Classification Report Heatmap — LightGBM')
plt.tight_layout()
plt.savefig('fig7c_classification_report.png', dpi=300)
plt.show()


## Figure 8: Summary Dashboard
Multi-panel summary of key pipeline results.

In [ ]:
# Figure 8: Summary Dashboard
fig = plt.figure(figsize=(18, 10))

# Best model accuracy
ax1 = fig.add_subplot(2, 3, 1)
final_acc = accuracy_score(y_test, y_pred)
ax1.text(0.5, 0.5,
         f'Best Model:\nLightGBM\n\nAccuracy: {final_acc*100:.2f}%',
         ha='center', va='center', fontsize=14, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
ax1.axis('off')
ax1.set_title('Best Model Summary', fontsize=12)

# Feature importances from LightGBM
ax2 = fig.add_subplot(2, 3, 2)
feat_imp = pd.Series(lgb_model.feature_importances_,
                     index=selected_features).sort_values(ascending=True)
ax2.barh(feat_imp.index, feat_imp.values, color='steelblue')
ax2.set_title('Feature Importance (LightGBM)', fontsize=12)
ax2.set_xlabel('Importance')

# Class balance pie
ax3 = fig.add_subplot(2, 3, 3)
after_counts_dash = pd.Series(y_res).value_counts().sort_index()
ax3.pie(after_counts_dash.values, labels=['No Diabetes', 'Diabetes'],
        autopct='%1.1f%%', colors=['steelblue', 'salmon'], startangle=90)
ax3.set_title('Class Balance After Oversampling', fontsize=12)

# ROC Curve
ax4 = fig.add_subplot(2, 3, 4)
ax4.plot(fpr_nb, tpr_nb, label=f'LightGBM (AUC={auc(fpr_nb, tpr_nb):.4f})', color='steelblue')
ax4.plot([0, 1], [0, 1], 'k--')
ax4.set_xlabel('FPR')
ax4.set_ylabel('TPR')
ax4.set_title('ROC Curve', fontsize=12)
ax4.legend(fontsize=9)

# Data pipeline row counts
ax5 = fig.add_subplot(2, 3, 5)
pipeline_stages = ['Raw Data', 'After NaN\nReplacement', 'After\nImputation',
                   'After IQR\nCleaning', 'After\nOversampling']
pipeline_counts = [len(df), len(df), len(df), len(X_clean), len(X_res)]
ax5.bar(pipeline_stages, pipeline_counts, color='teal', alpha=0.7)
ax5.set_title('Data Pipeline: Row Counts', fontsize=12)
ax5.set_ylabel('Number of Rows')
ax5.tick_params(axis='x', rotation=20)
for bar, val in zip(ax5.patches, pipeline_counts):
    ax5.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
             str(val), ha='center', fontsize=9)

# Confusion matrix
ax6 = fig.add_subplot(2, 3, 6)
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot(ax=ax6, colorbar=False)
ax6.set_title('Confusion Matrix (LightGBM)', fontsize=12)

plt.suptitle('Figure 8: Summary Dashboard — Diabetes Prediction Pipeline',
             fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('fig8_summary_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()
